# 05 - Preset Tuning for GPU Budget

This notebook explains how to tune memory/speed knobs without changing math semantics.

Preset intent:
- `cpu`: maximum compatibility and CPU execution
- `gpu`: keep memory and compute on CUDA for the tutorial default
- `hybrid`: available when you want CPU memory with CUDA compute


## Notebook Guide / 노트북 가이드

**EN**  This is an operational tuning notebook. It is short on purpose: the goal is to show what preset changes actually do.

**KO**  이 노트북은 운영/튜닝 관점의 짧은 노트북입니다. preset을 바꿨을 때 무엇이 달라지는지 빠르게 확인하는 것이 목적입니다.

- **EN** Read this after you already understand the basic compile/eval flow.
- **KO** 기본 compile/eval 흐름을 이해한 뒤에 보는 것이 좋습니다.
- **EN** Expected outcome: you should know how to adjust `chunk_size`, `dtype`, and truncation-related knobs without changing the high-level API.
- **KO** 기대 결과: high-level API는 유지한 채 `chunk_size`, `dtype`, truncation 관련 knob를 조절하는 법을 익혀야 합니다.


In [2]:
# EN: This notebook only needs the preset resolver and compiler, because the focus is configuration rather than training.
# KO: 이 노트북은 학습보다 설정 비교가 중심이므로 preset resolver와 compiler 위주로만 불러옵니다.

from pathlib import Path
import sys

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
sys.path.insert(0, str(root))

from src_tensor.api import resolve_preset, compile_expval_program
from src.pauli_surrogate_python import PauliRotation, CliffordGate, PauliSum

In [3]:
base = resolve_preset("gpu")
print("base:", base)

custom = resolve_preset(
    "gpu",
    overrides={
        "max_weight": 6,
        "dtype": "float32",
    },
)
print("custom:", custom)

base: TensorSurrogatePreset(memory_device='cuda', compute_device='cuda', dtype='float64', max_weight=1000000000, weight_x=1.0, weight_y=1.0, weight_z=1.0, chunk_size=10000000)
custom: TensorSurrogatePreset(memory_device='cuda', compute_device='cuda', dtype='float32', max_weight=6, weight_x=1.0, weight_y=1.0, weight_z=1.0, chunk_size=10000000)


## Example compile with targeted overrides

Common knobs:
- `max_weight`, `max_xy`: structural truncation strength
- `offload_steps`: whether to keep old sparse steps on CPU
- `step_device`, `stream_device`: storage vs compute devices


In [4]:
n_qubits = 2
circuit = [PauliRotation("ZZ", [0, 1], param_idx=0)]
obs = PauliSum(n_qubits)
obs.add_from_str("ZZ", 1.0, qubits=[0, 1])

program = compile_expval_program(
    circuit=circuit,
    observables=[obs],
    preset="gpu",
    preset_overrides={
        "max_weight": 6,
    },
)

print("program preset:", program.preset)

propagate: 100%|██████████| 1/1 [00:00<00:00,  6.77it/s]


[PPS Info] Propagation complete. Terms generated: 1
Starting zero-filtering on cuda...
[ZeroFilter] Initial pruning (diagonal terms only): 1 -> 1 terms kept (100.00000000%)
[ZeroFilter] Zero-filtering done. Starting back-propagation of keep mask through 1 steps...


zero-filter: 100%|██████████| 1/1 [00:00<00:00, 839.20it/s, step=0 rows=1 cols=1 nnz=0]

program preset: TensorSurrogatePreset(memory_device='cuda', compute_device='cuda', dtype='float64', max_weight=6, weight_x=1.0, weight_y=1.0, weight_z=1.0, chunk_size=10000000)


## Practical policy
1. Start from `gpu` defaults.
2. If memory is tight, reduce `max_weight` first.
3. If speed is too low and memory allows, reduce offloading / move step storage closer to compute device.
4. Always re-check error vs exact small-n baseline after tuning.
